In [1]:
import os
import pandas as pd
import numpy as np
import joblib
from pmdarima import ARIMA, auto_arima
from src.common.utils import get_root_directory, collate_array_elements, make_directory
from src.common.stats import root_mean_squared_percentage_error
import numpy as np
from src.preprocessing.data_loader import DataLoader
from sklearn.metrics import root_mean_squared_error, mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, precision_score, accuracy_score
import mlflow

In [2]:
# PARAMETERS TO CHANGE
split_size = 0.15
start_date = "01/01/2021"
end_date = "31/12/2023"
maxiter=500
n_periods=7
model_type = "ARIMA"
split_name = "validation"
experiment_name = f"{model_type}_train_{split_name}"
max_p = 7
max_q = 1
d = 1
test_p = 1
test_q = 1

In [3]:
root_dir = get_root_directory()
DL = DataLoader(root_dir)
DL.load_data()
DL.set_time_range(start_date=start_date, end_date=end_date)
if split_name=="validation":
    train, test = DL.split_data(split_size=split_size)
    train, val = DL.split_data(split_type="train_val",split_size=split_size)
    test = test["ETH_D_AvgPrc"]
    train = train["ETH_D_AvgPrc"]
    val = val["ETH_D_AvgPrc"]
    split = val[:-n_periods]
elif split_name=="test":
    train, test = DL.split_data(split_size=0.15)
    test = test["ETH_D_AvgPrc"]
    train = train["ETH_D_AvgPrc"]
    split = test[:-n_periods]

In [5]:
mlflow.set_tracking_uri(f"sqlite:///{root_dir}/mlruns/mlruns.db")
mlflow.set_experiment(experiment_name)

for p in range(1,max_p+1):
    for q in range(1,max_q+1):
        predictions = []
        train_recursive = list(train)
        order = (p, d, q)
        run_name = f"order_({p},{d},{q})"
        with mlflow.start_run(run_name=run_name):
            model = ARIMA(maxiter=maxiter, order=order)
            mlflow.log_params({"p": p,"d": d,"q": q,"maxiter":maxiter,"n_periods": n_periods, "prediction_start_date":split.index.to_list()[0].strftime('%d-%m-%Y')})
            for t in range(len(split)):
                model.fit(train_recursive)
                forecast = model.predict(n_periods=n_periods)
                predictions.append(forecast.tolist())
                train_recursive.append(split.iloc[t])
            df = pd.DataFrame(predictions, columns=[f"t{i+1}" for i in range(len(predictions[0]))])
            mse_list = []
            rmse_list = []
            rmspe_list = []
            mae_list = []
            mape_list = []
            accuracy_list  = []
            precision_list = []
            for i in range(len(df.columns)):
                df[f't{i+1}_prc_dir'] = df[f't{i+1}'].diff().apply(lambda x: 1 if x > 0 else -1)
                mse = mean_squared_error(val[i:-n_periods+i], df[f't{i+1}'])
                rmse = root_mean_squared_error(val[i:-n_periods+i],  df[f't{i+1}'])
                rmspe = root_mean_squared_percentage_error(val[i:-n_periods+i],  df[f't{i+1}'])
                mae = mean_absolute_error(val[i:-n_periods+i],  df[f't{i+1}'])
                mape = mean_absolute_percentage_error(val[i:-n_periods+i],  df[f't{i+1}'])
                accuracy = accuracy_score(val[i:-n_periods+i].diff().apply(lambda x: 1 if x > 0 else -1).dropna(), df[f't{i+1}_prc_dir'].dropna())
                precision = accuracy_score(val[i:-n_periods+i].diff().apply(lambda x: 1 if x > 0 else -1).dropna(), df[f't{i+1}_prc_dir'].dropna())
                mse_list.append(mse)
                rmse_list.append(rmse)
                rmspe_list.append(rmspe)
                mae_list.append(mae)
                mape_list.append(mape)
                accuracy_list.append(accuracy)
                precision_list.append(precision)
                avg_mse = sum(mse_list)/len(mse_list)
                avg_rmse = sum(rmse_list)/len(rmse_list)
                avg_mae = sum(mae_list)/len(mae_list)
                avg_mape = sum(mape_list)/len(mape_list)
                avg_accuracy = sum(accuracy_list)/len(accuracy_list)
                avg_precision = sum(precision_list)/len(precision_list)
                mlflow.log_metrics({
                    "mse_daily":mse_list[i], 
                    "rmse_daily":rmse_list[i],
                    "rmspe_daily":rmspe_list[i], 
                    "mae_daily":mae_list[i], 
                    "mape_daily":mape_list[i], 
                    "accuracy_daily":accuracy_list[i],
                    "precision_daily":precision_list[i],
                    "avg_mse": avg_mse,
                    "avg_rmse": avg_rmse,
                    "avg_mae": avg_mae,
                    "avg_mape": avg_mape,
                    "avg_accuracy":avg_accuracy,
                    "avg_precision":avg_precision
                }, step=(i+1))
        mlflow.end_run()
        results_file = f"{run_name}_{split_name}"
        path = str(os.path.join(root_dir,"mlruns",model_type))
        df.to_csv(str(os.path.join(path,results_file)), index=False)
        print(f"Completed order: {order}")

Completed order: (1, 1, 1)
Completed order: (2, 1, 1)
Completed order: (3, 1, 1)
Completed order: (4, 1, 1)
Completed order: (5, 1, 1)
Completed order: (6, 1, 1)
Completed order: (7, 1, 1)


In [7]:
mlflow.set_tracking_uri(f"sqlite:///{root_dir}/mlruns/mlruns.db")
mlflow.set_experiment(experiment_name)

predictions = []
train_recursive = list(train)
order = (test_p, d, test_q)
run_name = f"order_({test_p},{d},{test_q})"
with mlflow.start_run(run_name=run_name):
    model = ARIMA(maxiter=maxiter, order=order)
    mlflow.log_params({"p": test_p,"d": d,"q": test_q,"maxiter":maxiter,"n_periods": n_periods, "prediction_start_date":split.index.to_list()[0].strftime('%d-%m-%Y')})
    for t in range(len(split)):
        model.fit(train_recursive)
        forecast = model.predict(n_periods=n_periods)
        predictions.append(forecast.tolist())
        train_recursive.append(split.iloc[t])
    df = pd.DataFrame(predictions, columns=[f"t{i+1}" for i in range(len(predictions[0]))])
    mse_list = []
    rmse_list = []
    rmspe_list = []
    mae_list = []
    mape_list = []
    accuracy_list  = []
    precision_list = []
    for i in range(len(df.columns)):
        df[f't{i+1}_prc_dir'] = df[f't{i+1}'].diff().apply(lambda x: 1 if x > 0 else -1)
        mse = mean_squared_error(test[i:-n_periods+i], df[f't{i+1}'])
        rmse = root_mean_squared_error(test[i:-n_periods+i],  df[f't{i+1}'])
        rmspe = root_mean_squared_percentage_error(test[i:-n_periods+i],  df[f't{i+1}'])
        mae = mean_absolute_error(test[i:-n_periods+i],  df[f't{i+1}'])
        mape = mean_absolute_percentage_error(test[i:-n_periods+i],  df[f't{i+1}'])
        accuracy = accuracy_score(test[i:-n_periods+i].diff().apply(lambda x: 1 if x > 0 else -1).dropna(), df[f't{i+1}_prc_dir'].dropna())
        precision = accuracy_score(test[i:-n_periods+i].diff().apply(lambda x: 1 if x > 0 else -1).dropna(), df[f't{i+1}_prc_dir'].dropna())
        mse_list.append(mse)
        rmse_list.append(rmse)
        rmspe_list.append(rmspe)
        mae_list.append(mae)
        mape_list.append(mape)
        accuracy_list.append(accuracy)
        precision_list.append(precision)
        avg_mse = sum(mse_list)/len(mse_list)
        avg_rmse = sum(rmse_list)/len(rmse_list)
        avg_mae = sum(mae_list)/len(mae_list)
        avg_mape = sum(mape_list)/len(mape_list)
        avg_accuracy = sum(accuracy_list)/len(accuracy_list)
        avg_precision = sum(precision_list)/len(precision_list)
        mlflow.log_metrics({
            "mse_daily":mse_list[i], 
            "rmse_daily":rmse_list[i],
            "rmspe_daily":rmspe_list[i], 
            "mae_daily":mae_list[i], 
            "mape_daily":mape_list[i], 
            "accuracy_daily":accuracy_list[i],
            "precision_daily":precision_list[i],
            "avg_mse": avg_mse,
            "avg_rmse": avg_rmse,
            "avg_mae": avg_mae,
            "avg_mape": avg_mape,
            "avg_accuracy":avg_accuracy,
            "avg_precision":avg_precision
        }, step=(i+1))
mlflow.end_run()
results_file = f"{run_name}_{split_name}"
path = str(os.path.join(root_dir,"mlruns",model_type))
df.to_csv(str(os.path.join(path,results_file)), index=False)
print(f"Completed order: {order}")

Completed order: (1, 1, 1)
